# Análise do Winning the Race — America's AI Action Plan

Este notebook integra uma análise acadêmica de rigor, desenvolvida no âmbito de uma pesquisa em Relações Internacionais dedicada ao estudo comparado das estratégias nacionais de inteligência artificial. Seu objeto específico é o documento "Winning the Race: America's AI Action Plan", publicado pela Casa Branca, que estabelece a estratégia dos Estados Unidos para a liderança global em inteligência artificial.

A análise aqui conduzida busca compreender, de modo simultaneamente **quantitativo e qualitativo**, a linguagem empregada pelo documento — os termos, categorias, ênfases retóricas, valores e prioridades estratégicas que estruturam o texto —, de modo a identificar os marcos conceituais e os campos semânticos por meio dos quais os Estados Unidos formula sua política de inteligência artificial.

A partir dessa leitura, pretende-se **posicionar o documento internacionalmente**, comparando a linguagem e as escolhas discursivas de os Estados Unidos com o vocabulário e as ênfases adotados pelos demais países e blocos contemplados neste projeto (Brasil, China, Estados Unidos, Europa e Índia), de modo a mapear convergências, divergências e posicionamentos estratégicos distintivos no debate internacional sobre desenvolvimento e governança de inteligência artificial.

As análises e visualizações produzidas neste notebook seguem as diretrizes metodológicas da **Skill02** (Análise e Visualização Gráfica de Documentos): baseiam-se exclusivamente no campo `texto_completo` do JSON de extração correspondente a este documento, produzido na etapa anterior (Skill01), adotam rigor acadêmico integral na leitura do texto-fonte, evitam generalizações, simplificações e inferências não fundamentadas no texto original, e cada visualização construída é acompanhada de sua respectiva descrição, leitura, interpretação e eventuais limitações metodológicas.


## Análise de Vocabulário — Termos mais Frequentes

Esta seção aplica a **Skill02** (Análise e Visualização Gráfica de Documentos) em conjunto com seu complemento especializado, a **Skill 02_Análise_Vocab_A** (Análise de Vocabulário e Termos), ao JSON `americas_ai_action_plan.json`, produzido na etapa anterior (Skill01).

**Protocolo de uso do JSON.** Conforme exigido pela Skill02, apenas os campos `titulo`, `pais_ou_bloco` e `texto_completo` são utilizados nesta análise. Os campos `elementos_descartados`, `data_extracao`, `data_publicacao` e `fonte` não entram, em nenhuma hipótese, na contagem ou na interpretação do vocabulário.

**Idioma identificado.** O documento está integralmente redigido em **inglês**. Todas as listas de palavras funcionais e as regras de normalização morfológica aplicadas abaixo correspondem ao inglês, e não ao português ou a qualquer outro idioma.

**Etapas metodológicas aplicadas** (ver célula de código a seguir para a implementação exata):

1. **Extração bruta** de todos os termos de `texto_completo`, sem qualquer corte prévio de conteúdo.
2. **Pré-processamento do texto corrido, antes da tokenização**, fundamentado em identidade de referente (não em conveniência estatística):
   - `"U.S."` foi expandido para `"United States"`, e a expressão composta `"United States"` foi tratada como uma única unidade lexical (`united_states`) — mesma sigla e forma por extenso designando o mesmo país, estrutura análoga ao par "IA"/"Inteligência Artificial" citado na própria Skill 02_Análise_Vocab_A.
   - A expressão composta `"Artificial Intelligence"` foi unificada ao termo `"AI"`, pelo mesmo critério.
   - O subtítulo estrutural fixo **"Recommended Policy Actions"**, que se repete mecanicamente 30 vezes ao longo do documento — uma vez antes de cada lista de ações recomendadas, sem nenhuma ocorrência com sentido distinto —, foi removido da contagem lexical. Trata-se de um rótulo editorial repetitivo, e não de vocabulário empregado organicamente no texto corrido (mesma natureza dos cabeçalhos de página já descartados na Skill01).
3. **Tokenização**: sequências de letras (`[A-Za-z]+`) foram extraídas como tokens; qualquer caractere não alfabético (espaço, hífen, apóstrofo, ponto, parênteses, barra) foi tratado como separador. Compostos com hífen (ex.: *open-source*, *AI-related*, *large-scale*) foram, por consequência, fragmentados em suas palavras componentes — regra única e uniforme, aplicada sem exceção a todos os hífens do texto, para evitar qualquer critério ad hoc de "quais hifens preservar" (essa opção metodológica é discutida como limitação ao final).
4. **Remoção (exclusão)** de:
   - Palavras funcionais do inglês (artigos, preposições, conjunções, pronomes, verbos auxiliares/modais e conectores discursivos genéricos como *also*, *including*, *etc.*) — lista declarada integralmente na célula de código e no registro persistente;
   - Tokens de um único caractere, resíduos não semânticos da tokenização de abreviações pontuadas (ex.: "e.g." gera fragmentos soltos "e"/"g") e de possessivos (ex.: "nation's" gera o fragmento solto "s");
   - O subtítulo estrutural "Recommended Policy Actions" (ver item 2).
5. **Normalização/agrupamento** de variantes que designam exatamente o mesmo referente: 35 pares singular/plural de substantivos (ex.: *system*/*systems*, *agency*/*agencies*) e 12 famílias de conjugações verbais (ex.: *lead*/*leads*/*leading*/*led*). Pares com grafia semelhante mas classe gramatical ou referente distintos foram deliberadamente **mantidos separados** — por exemplo, *America* (topônimo) / *American* (adjetivo) / *Americans* (substantivo, pessoas); *research* / *researchers*; *development* / *develop*; *security* / *secure*; e, criticamente, *state(s)* no sentido de estados subnacionais dos EUA foi mantido separado de "United States", por serem referentes distintos apesar da coincidência lexical parcial.
6. **Critério de corte declarado:** os **25 termos de maior frequência absoluta** após as etapas 1–5, em ordem decrescente. Frequência absoluta é adequada aqui porque esta é uma análise de um único documento — a normalização relativa entre documentos de tamanhos distintos, exigida pela Skill02, aplica-se apenas à pasta "Análise Conjunta".

O registro completo, persistente e auditável de todas as remoções e agrupamentos está salvo em [`registro_vocabulario_americas_ai_action_plan.md`](./registro_vocabulario_americas_ai_action_plan.md), nesta mesma pasta, e deve ser consultado — e atualizado de forma cumulativa — antes de qualquer nova visualização de vocabulário sobre este mesmo documento.


In [ ]:
import json, re
from collections import Counter
import pandas as pd

with open("americas_ai_action_plan.json", encoding="utf-8") as f:
    doc = json.load(f)

titulo, pais, texto = doc["titulo"], doc["pais_ou_bloco"], doc["texto_completo"]
print(f"Documento: {titulo} | País/bloco: {pais} | Caracteres em texto_completo: {len(texto)}")

STOPWORDS_EN = set("""
a an the
and or but nor so yet if because while although that which who whom whose when where how than whether as
this these those it its they their them theirs he she his her him himself herself itself themselves
we our ours us you your yours i my mine
is are was were be been being am
has have had having
do does did doing
will would shall should can could may might must
to of in on at by for with from into through across throughout under over without within among between via per about upon toward towards
not no nor
also more most many much very only further therefore thus however moreover
there here
such other others any all both each every some
including include includes included
etc eg ie
""".split())

STRUCTURAL_PHRASES_REMOVIDAS = [r'\bRecommended Policy Actions\b']

NOUN_MERGES = {
    "systems": ["system", "systems"], "models": ["model", "models"], "agencies": ["agency", "agencies"],
    "technology": ["technology", "technologies"], "programs": ["program", "programs"], "workers": ["worker", "workers"],
    "controls": ["control", "controls"], "standards": ["standard", "standards"], "capabilities": ["capability", "capabilities"],
    "tools": ["tool", "tools"], "actions": ["action", "actions"], "risks": ["risk", "risks"],
    "developers": ["developer", "developers"], "initiatives": ["initiative", "initiatives"], "stakeholders": ["stakeholder", "stakeholders"],
    "frameworks": ["framework", "frameworks"], "resources": ["resource", "resources"], "occupations": ["occupation", "occupations"],
    "assessments": ["assessment", "assessments"], "evaluations": ["evaluation", "evaluations"], "regulations": ["regulation", "regulations"],
    "values": ["value", "values"], "threats": ["threat", "threats"], "vulnerabilities": ["vulnerability", "vulnerabilities"],
    "centers": ["center", "centers"], "innovation": ["innovation", "innovations"], "needs": ["need", "needs"],
    "skills": ["skill", "skills"], "countries": ["country", "countries"], "efforts": ["effort", "efforts"],
    "employers": ["employer", "employers"], "industry": ["industry", "industries"], "partners": ["partner", "partners"],
    "sector": ["sector", "sectors"], "services": ["service", "services"], "sources": ["source", "sources"],
    "states (subnacionais, distinto de United States)": ["state", "states"],
}

VERB_MERGES = {
    "lead / led": ["lead", "leads", "leading", "led"],
    "develop": ["develop", "develops", "developing", "developed"],
    "create": ["create", "creates", "creating", "created"],
    "build": ["build", "builds", "building", "built"],
    "establish": ["establish", "establishes", "establishing", "established"],
    "ensure": ["ensure", "ensures", "ensuring", "ensured"],
    "promote": ["promote", "promotes", "promoting", "promoted"],
    "expand": ["expand", "expands", "expanding", "expanded"],
    "support": ["support", "supports", "supporting", "supported"],
    "make": ["make", "makes", "making", "made"],
    "require": ["require", "requires", "requiring", "required"],
    "use": ["use", "uses", "used", "using"],
}

def preprocess(text):
    text = text.replace("U.S.", "United States")
    text = re.sub(r'\bArtificial Intelligence\b', 'AI', text, flags=re.I)
    for pat in STRUCTURAL_PHRASES_REMOVIDAS:
        text = re.sub(pat, ' ', text)
    return text

texto_tratado = preprocess(texto)
tokens_brutos = re.findall(r"[A-Za-z]+", texto_tratado)
tokens_lower = [t.lower() for t in tokens_brutos]

tokens_merged_us = []
i = 0
while i < len(tokens_lower):
    if tokens_lower[i] == "united" and i + 1 < len(tokens_lower) and tokens_lower[i + 1] == "states":
        tokens_merged_us.append("united_states"); i += 2
    else:
        tokens_merged_us.append(tokens_lower[i]); i += 1

tokens_filtrados = [t for t in tokens_merged_us if len(t) > 1 and t not in STOPWORDS_EN]

canon = {}
for label, variants in NOUN_MERGES.items():
    for v in variants:
        canon[v] = label
for label, variants in VERB_MERGES.items():
    for v in variants:
        canon[v] = label
canon["ai"] = "AI"
canon["united_states"] = "United States"
canon["doc"] = "DOC"

tokens_finais = [canon.get(t, t) for t in tokens_filtrados]
freq = Counter(tokens_finais)

print(f"Tokens brutos: {len(tokens_brutos)} | após remoção de funcionais/resíduos: {len(tokens_filtrados)} | termos distintos no vocabulário final: {len(freq)}")

DISPLAY_CAPITALIZE = {"american": "American", "america": "America", "trump": "Trump"}

CORTE = 25
top_termos_raw = freq.most_common(CORTE)
top_termos = [(DISPLAY_CAPITALIZE.get(t, t), n) for t, n in top_termos_raw]

tabela_top = pd.DataFrame(top_termos, columns=["termo", "frequência"])
tabela_top.index = tabela_top.index + 1
tabela_top


In [ ]:
import matplotlib.pyplot as plt

termos_plot = [t for t, _ in top_termos][::-1]
freqs_plot = [n for _, n in top_termos][::-1]

fig, ax = plt.subplots(figsize=(9, 10))
barras = ax.barh(termos_plot, freqs_plot, color="#1f4e79", height=0.68)

ax.set_xlabel("Frequência absoluta (nº de ocorrências em texto_completo)", fontsize=11)
ax.set_title(
    "Termos mais frequentes — America's AI Action Plan (EUA, 2025)\n"
    f"Top {CORTE} termos após remoção de palavras funcionais e normalização de variantes",
    fontsize=12.5, fontweight="bold", loc="left", pad=14
)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", labelsize=10.5)
ax.tick_params(axis="x", labelsize=9.5)
ax.set_axisbelow(True)
ax.xaxis.grid(True, color="#d9d9d9", linewidth=0.8)

for bar, val in zip(barras, freqs_plot):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height() / 2, str(val),
            va="center", ha="left", fontsize=9.5, color="#333333")

ax.set_xlim(0, max(freqs_plot) * 1.12)
fig.text(0.02, -0.01,
         "Fonte: texto_completo de americas_ai_action_plan.json (Skill01), tratado segundo a Skill02 / Skill 02_Análise_Vocab_A.",
         fontsize=8, color="#666666")
fig.tight_layout()
plt.show()


## Leitura, Interpretação e Limitações do Gráfico

**O que foi construído.** Um gráfico de barras horizontais com os 25 termos de maior frequência absoluta em `texto_completo`, após remoção de palavras funcionais/resíduos e normalização das variantes descritas acima. Cada barra representa a contagem final (já consolidada) de um termo ou família de termos equivalentes.

**Como ler o gráfico.** O eixo vertical lista os termos, em ordem decrescente de frequência (o termo mais frequente no topo); o eixo horizontal mede o número absoluto de ocorrências no documento. O rótulo numérico ao final de cada barra é a contagem exata.

**Interpretação à luz do documento.**
- **"AI" (269 ocorrências)** domina o vocabulário por larga margem — mais de quatro vezes a frequência do segundo termo mais comum —, o que é esperado e consistente com o objeto do documento: um plano de ação inteiramente dedicado à inteligência artificial. Esse resultado, isoladamente, tem baixo valor discriminante (é óbvio que um "AI Action Plan" fala predominantemente de "AI"); o valor analítico do gráfico está principalmente nos termos que disputam a segunda posição em diante, pois eles revelam **como** o documento enquadra a IA.
- **"lead / led" (61 ocorrências)** aparece em segundo lugar, mas majoritariamente pela fórmula institucional recorrente **"Led by [órgão federal]..."**, que abre a maior parte das ações de política recomendada (48 das 61 ocorrências, antes do agrupamento verbal, correspondem à forma "led"). Isso não reflete um tema discursivo de "liderança" em sentido retórico, mas sim a arquitetura de governança do documento: cada ação é explicitamente atribuída a um órgão federal responsável (DOC, DOD, DOE, NIST, OSTP, CAISI, entre outros). É, em si, um achado relevante para uma leitura de Relações Internacionais: o plano se estrutura como um conjunto de mandatos interagenciais, não como uma declaração de princípios genérica.
- **"systems" (55), "models" (33), "technology" (40), "data" (37), "infrastructure" (34)** confirmam o vocabulário técnico-operacional esperado de um plano de infraestrutura e adoção de IA.
- **"United States" (50) e "DOC" (50)**, empatados, e **"federal" (49), "national" (44), "government" (27)** evidenciam a centralidade do aparato federal americano como agente da política — nenhuma empresa nomeada entra no top 25, e o vocabulário se concentra nas instituições públicas responsáveis pela execução.
- **"security" (42)** no top 10 reforça o enquadramento do documento como uma questão de segurança nacional tanto quanto de política industrial — consistente com a moldura retórica da "corrida" (*race*) já presente no título e na introdução.
- **"American" (36) e "America" (25)**, mantidos propositalmente separados (ver metodologia), somados ultrapassariam "federal" — juntos, evidenciam a ênfase retórica nacionalista/identitária do documento, distinta da referência técnico-jurídica "United States"/"U.S.".
- **"ensure" (32), "build" (27)**, e outros verbos de ação normalizados que ficaram logo abaixo do corte de 25 (*develop*, *establish*, *expand*, *support*, *promote* — todos presentes no vocabulário tratado, consultável na tabela de frequências completa gerada pela célula de código), formam um campo semântico de verbos típicos de um documento programático/normativo.

**Ajustes possíveis para aprimorar a visualização.**
- Construir um segundo gráfico separando explicitamente os termos que são nomes de órgãos/siglas (DOC, DOD, NIST, DOE, OSTP, CAISI...) dos termos temático-conceituais (security, infrastructure, systems...), para não misturar, na mesma leitura, "quem executa" e "o que é discutido".
- Segmentar a contagem por Pilar (I, II, III) do documento, permitindo comparar o vocabulário dominante em "Acelerar a Inovação em IA" vs. "Construir Infraestrutura" vs. "Diplomacia e Segurança Internacional" — não realizado aqui porque o pedido do usuário foi por uma visão geral do documento como um todo, não uma análise por seção.
- Reportar, ao lado da frequência absoluta, a frequência relativa (percentual do total de tokens tratados) para leitores que desejem comparar a proeminência relativa dos termos sem depender do tamanho do documento.

**Limitações e fraquezas metodológicas.**
- **Polissemia não desambiguada:** a contagem é lexical, não semântica. Por exemplo, o token "generation" agrega tanto o sentido de "geração de energia" (*power generation*) quanto o de "nova geração tecnológica" (*next-generation*) — usos distintos contados como um único termo. Esse termo específico não entrou no top 25, mas o mesmo tipo de limitação pode afetar, em menor grau, outros termos polissêmicos não identificados manualmente.
- **Fragmentação de compostos com hífen:** por aplicar uma regra única de tokenização (hífen sempre como separador), expressões tecnicamente relevantes como "open-source" e "open-weight" foram fragmentadas em "open"/"source"/"weight", dissolvendo-se no termo genérico "open" (que não chegou ao top 25). Uma análise futura poderia tratar essas expressões como unidades compostas explícitas, à custa de introduzir uma lista de exceções que precisaria ser justificada termo a termo.
- **Cobertura da normalização morfológica:** a checagem de pares singular/plural e conjugações verbais foi realizada de forma sistemática entre os termos com frequência ≥ 4 no vocabulário já filtrado (a faixa plausível de disputar o corte de 25) — não houve lematização exaustiva de toda a cauda longa do vocabulário (mais de 1.700 termos distintos). Isso é adequado para o gráfico produzido, mas não geraria automaticamente o mesmo resultado caso o critério de corte fosse ampliado para, por exemplo, os 100 termos mais frequentes.
- **"lead/led" como categoria mista:** embora a fusão das conjugações verbais siga a regra declarada pela Skill 02_Análise_Vocab_A, o resultado agrega, sob um único rótulo, um uso quase inteiramente formulaico ("Led by X") a usos esporádicos de sentido mais amplo de liderança/pioneirismo (ex.: "leading pioneer", "must lead the world"). Uma inspeção qualitativa (não apenas quantitativa) seria necessária para separar essas duas funções discursivas com precisão.

A metodologia completa — incluindo a lista integral de palavras removidas, os 35 agrupamentos de substantivos e as 12 famílias verbais normalizadas, e os pares deliberadamente não agrupados — está documentada e versionada em [`registro_vocabulario_americas_ai_action_plan.md`](./registro_vocabulario_americas_ai_action_plan.md).
